# Heston stochastic local volatility (SLV)

How the stochastic local vol model works, step by step, on TICKER1:

1. **why**: local vol has the right marginals and the wrong dynamics, Heston the opposite;
2. **the model**: Heston variance times a leverage function, and why the leverage is fixed by Gyöngy's theorem;
3. **the calibration, by hand**: local vol, Heston, then the leverage built on the particles step by step, checked
   against `HestonSLVModel`; the smile repriced;
4. **the mixing factor**: one dial between local vol and Heston dynamics, with the same vanillas;
5. **an exotic**: a barrier under the whole family;
6. **Monte Carlo**: how well the vanillas are repriced (noise and a systematic error), greeks by AAD and the risk
   convention;
7. **the limits** of the model.

Docs: *Models and Pricing* (Models: Heston stochastic local volatility), *Architecture* (Models). Read after
`local_vol.ipynb` and `heston.ipynb`.

In [ ]:
import os
import time
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq
from scipy.optimize import brentq
from scipy.stats import norm

## The calibration in a nutshell

**Input**: the implied vol surface (8 maturities $\times$ 9 strikes) and one number chosen by the user, the **mixing
factor** $\eta \in (0, 1]$.
**Output**: the Heston parameters, the local vol surface, and a **leverage function** $L(t, x)$ that makes the model
reprice the surface exactly, as local vol does, while keeping a stochastic variance.

The steps:

1. **Local vol** from the surface (Dupire–Gatheral, `local_vol.ipynb`): the target local variance $\sigma_{\mathrm{LV}}^2(t, x)$.
2. **Heston** calibrated to the surface (`heston.ipynb`), with the vol of vol scaled by $\eta$: the stochastic variance $v_t$.
3. **Leverage** $L^2(t, x) = \sigma_{\mathrm{LV}}^2(t, x)\,/\,\mathbb{E}[v_t \mid X_t = x]$: no optimisation, a formula. But $\mathbb{E}[v_t \mid X_t = x]$ depends on the
   joint distribution of $(X_t, v_t)$, which depends on the leverage at earlier times. So it is built
   **forward in time on simulated particles**: at each time step, estimate $\mathbb{E}[v \mid X]$ from the particles,
   compute $L$, move the particles one step; the estimates are **stored** (a table: 50 time rows a year $\times$ 41 nodes
   in $\ln X$).
4. **Check**: the simulated smile is the market one.

```
surface ──1──> sigma_LV(t, x)                           (exact, Dupire)
surface ──2──> Heston (v0, kappa, theta, eta xi, rho)    (least squares)
for each time row t_k (calibration particles, own seed):
    particles (X, v) ──3──> E[v | X] (smoother) ──> store; L = sigma_LV / sqrt(E[v | X]) ──> step (X, v) to t_{k+1}
pricing: L = sigma_LV(market) / sqrt(E[v | X] read from the table) on the pricing paths
```

Two speeds: Heston (step 2) now and then, `calibrate_leverage` (step 3, a few seconds) whenever the surface moves.
The table is the model's calibrated parameter, like the Heston parameters: the greeks to the market go through it
(the adjoint of step 3, `equity_models.ipynb`).

## 1. Why stochastic local vol
The two previous notebooks end with complementary limits:
- **local vol** reprices every vanilla, but its vol is a deterministic function of the spot: the future smile flattens,
  the vol has no randomness of its own;
- **Heston** has a random vol with a forward smile that keeps its skew, but five parameters do not reprice the surface
  (many vol points off at the short maturities).

An exotic should be priced by a model that reprices the vanillas that hedge it **and** has a realistic dynamics. SLV
puts the two together: Heston dynamics, corrected by a function of time and spot so that the marginals are exactly
those of local vol.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
vs = eq.vol_surface
S0 = float(eq.spot)
T_NODES = np.array(vs.maturity, dtype=float)
K_NODES = np.array(vs.strike, dtype=float)
IV = vs.volatility_matrix.numpy()
F_NODES = np.array([float(eq.forward(float(T))) for T in T_NODES])


def date_of(T):
    return EVALUATION_DATE + timedelta(days=max(1, int(round(T * 365))))


def bs_call(k, T, s):
    # undiscounted Black-Scholes call per unit forward
    sd = s * np.sqrt(T)
    d1 = -np.log(k) / sd + 0.5 * sd
    return norm.cdf(d1) - k * norm.cdf(d1 - sd)


def implied(call, k, T):
    # implied vol of an undiscounted call per unit forward (inverted on the OTM side)
    otm = call - (1.0 - k) if k < 1 else call
    f = (lambda s: bs_call(k, T, s) - (1.0 - k) - otm) if k < 1 else (lambda s: bs_call(k, T, s) - otm)
    try:
        return brentq(f, 1e-4, 4.0)
    except ValueError:
        return np.nan


def mc_smile(x, ks, T):
    # implied vols of calls on simulated forward ratios x (mean ~1) at strikes ks
    x = x / x.mean()
    return np.array([implied(np.mean(np.maximum(x - k, 0.0)), k, T) for k in ks])


t0 = time.time()
heston_model = tq.HestonModel.calibrate(eq)          # model instance: parameters, report, calibration Jacobian
heston = heston_model.process(eq)                   # bound to the equity: closed forms and simulation
lv = tq.LocalVolatilityModel().process(eq)          # local vol: no parameters, built from the surface
print(f"Heston and local vol built in {time.time() - t0:.1f} s")
print("Heston fit, max error per maturity (vol points):",
      dict(zip(np.round(T_NODES, 2), heston_model.report["max_error_volpts"].round(1))))

## 2. The model
On the forward ratio $X = S / F(t)$:
$$\frac{\mathrm{d}X_t}{X_t} = L(t, X_t)\,\sqrt{v_t}\,\mathrm{d}W^1_t,\qquad
\mathrm{d}v_t = \kappa(\theta - v_t)\,\mathrm{d}t + \eta\,\xi\sqrt{v_t}\,\mathrm{d}W^2_t,\qquad \mathrm{d}\langle W^1, W^2\rangle_t = \rho\,\mathrm{d}t .$$
The instantaneous variance of the stock is $L^2(t, X_t)\,v_t$: the Heston variance, rescaled by a deterministic function of
time and spot.

**Which leverage reprices the vanillas.** Gyöngy's theorem (Step C of `local_vol.ipynb`): a process with a random
instantaneous variance has the same one-dimensional distributions (hence the same vanilla prices at every maturity) as
the local vol model whose local variance is the **expected instantaneous variance given the spot**:
$$\sigma_{\mathrm{loc}}^2(t, x) = \mathbb{E}\big[L^2(t, X_t)\,v_t \,\big|\, X_t = x\big] = L^2(t, x)\;\mathbb{E}[v_t \mid X_t = x].$$
For the SLV model to have the market marginals this must be the market local vol, hence
$$\boxed{\;L^2(t, x) = \frac{\sigma_{\mathrm{LV}}^2(t, x)}{\mathbb{E}[v_t \mid X_t = x]}\;}$$
**Intuition.** On the paths that are at $x$ at time $t$, the Heston variance has some average. If it is too high compared
with the vol the market wants there, $L < 1$ scales it down; if too low, $L > 1$ scales it up. The **shape** of the
fluctuations (vol of vol, correlation, the forward smile) stays Heston's; only their **level**, as a function of the
spot, is corrected.

**Two limits** that explain the mixing factor:
- **Heston market**: if the surface came from a Heston model, the local vol is exactly $\sqrt{\mathbb{E}[v \mid X]}$ and $L = 1$;
- **$\eta \to 0$**: no vol of vol, $v_t$ is deterministic, $\mathbb{E}[v_t \mid X] = v_t$ and $L\sqrt{v_t} = \sigma_{\mathrm{LV}}$: the model *is* local vol.

So $\eta$ moves the model from local vol ($\eta \to 0$) to Heston dynamics ($\eta = 1$), **always with the same vanilla prices**.

**The circularity.** $\mathbb{E}[v_t \mid X_t = x]$ is a property of the joint distribution of $(X_t, v_t)$, which depends on the leverage
at all earlier times: the SDE depends on the law of its own solution (a McKean SDE). It is solved forward in time on
a cloud of simulated paths, the **particle method** (Guyon and Henry-Labordère, 2012; van der Stoep, Grzelak and
Oosterlee, 2014).

## 3. Calibration, step by step

### Step 1: the local vol (the target)
The local vol of the surface, exactly as in `local_vol.ipynb`; it reprices the vanillas by construction.

### Step 2: the Heston variance
The Heston parameters calibrated to the whole surface (`heston.ipynb`), with the vol of vol scaled by $\eta$. Here
$\eta = 1$: the calibrated Heston dynamics. The variance paths do not depend on the leverage (the leverage only acts on
$X$), but which variance a path has *at a given spot* does.

In [ ]:
p = {k: float(v) for k, v in heston.parameters.items()}
print(pd.Series(p).round(4).to_string())
print(f"Feller 2 kappa theta - xi^2 = {heston.feller:.3f}")

### Step 3: the leverage on the particles, one time step by hand
The calibration (`calibrate_leverage`, 50k particles of its own) has, at the start of each time row, $n$ particles
$(X^{(m)}, v^{(m)})$. It:
1. **estimates $\mathbb{E}[v \mid \ln X]$** by a smoother: 41 nodes evenly spaced between the 0.1% and 99.9% quantiles of $\ln X$ (widened
   by 1% of the range); each particle splits its weight between its two neighbouring nodes linearly (a tent kernel), so
   node $j$ gets $\mathbb{E}_j = \sum_m w_{mj} v^{(m)} / \sum_m w_{mj}$; sparse nodes are shrunk to the overall mean variance (a
   pseudo-count of $0.05\,n/41$ particles); **stores** the nodes and $\mathbb{E}_j$;
2. **computes the leverage** of each particle, $L = \sigma_{\mathrm{LV}}(t, \ln X) / \sqrt{\mathbb{E}[v \mid \ln X]}$ (read back with the same
   linear weights, floored at $10^{-4}$, a 1% vol), capped at 10;
   the smoother averages over a node spacing $h$, a bias of order $h^2$ (about $-0.6$ vol points at 3Y ATM with 41
   nodes): the estimate read back is the Richardson combination $(4\,\mathbb{E}^{h} - \mathbb{E}^{2h})/3$ of the estimates on the 41
   nodes and on every other node (21), which removes the leading term (about $-0.2$);
3. **moves the particles** one step: the Heston QE step for $v$, the log step for $X$ with $L$ frozen on the step, with the
   exact martingale correction (as in `heston.ipynb`).

The model stores **two tables** on the same nodes: the corrected one (`leverage_table["price_values"]`), which gives
the prices, and the 41-node one without correction (the parameter `expectation`), which gives the greeks (section 6).
The pricing paths read them (linear in $\ln X$ on the nodes, linear in time between rows). Below, the step
that starts at $t = 1$: the pricing particles of `HestonSLVProcess.simulate` (which also returns the variance and the
leverage used on each step), the leverage by hand from the stored table, and the smoother by hand on the pricing
particles against the stored (calibration) estimate.

In [ ]:
slv_model = tq.HestonSLVModel(heston_model, mixing=1.0).calibrate_leverage(eq)   # the E[v | X] table
slv = slv_model.process(eq)
STEPS = 50                                                          # per year
grid = np.arange(1, 3 * STEPS + 1) / STEPS                         # to 3Y
N_PATHS = 40_000
dw = tf.random.stateless_normal([N_PATHS, len(grid), 2], seed=[5, 0], dtype=tf.float64)
t0 = time.time()
X, V, LEV = (a.numpy() for a in slv.simulate(grid, dw))
print(f"{N_PATHS} particles, {len(grid)} steps: {time.time() - t0:.1f} s")

i = STEPS                                       # the step from t = 1.00 to 1.02: it uses the particles at t = 1
t_start = grid[i - 1]
y, v = np.log(X[:, i - 1]), V[:, i - 1]
row = int(round(t_start * slv_model.settings["steps_per_year"]))   # the table row at t = 1
nodes_y = slv_model.leverage_table["nodes"][row]
price_row = slv_model.leverage_table["price_values"][row]      # corrected calibration: 41 + 21 estimates
n, n_nodes = len(y), len(nodes_y)
stored, stored_coarse = price_row[:n_nodes], price_row[n_nodes:]


def tent_read(values, nodes):
    pos = np.clip((y - nodes[0]) / (nodes[-1] - nodes[0]) * (len(nodes) - 1), 0, len(nodes) - 1)
    j = np.clip(np.floor(pos).astype(int), 0, len(nodes) - 2)
    w = pos - j
    return values[j] * (1 - w) + values[j + 1] * w, j, w


# 1. the stored estimates read at the pricing particles (tent weights), Richardson combination
fine, j, w = tent_read(stored, nodes_y)
coarse = tent_read(stored_coarse, nodes_y[::2])[0]
cond_v = (4 * fine - coarse) / 3
# 2. leverage
F_t = float(eq.forward(float(t_start)))
sigma_lv = lv.sigma_loc(np.full(n, t_start), F_t * np.exp(y)).numpy()
lev_hand = np.minimum(sigma_lv / np.sqrt(np.maximum(cond_v, 1e-4)), 10.0)
print(f"max |leverage by hand - HestonSLVModel| = {np.abs(lev_hand - LEV[:, i]).max():.1e}")
# the smoother by hand on the pricing particles (same nodes): close to the stored estimate (other particles)
num = np.bincount(j, v * (1 - w), n_nodes) + np.bincount(j + 1, v * w, n_nodes)
den = np.bincount(j, 1 - w, n_nodes) + np.bincount(j + 1, w, n_nodes)
shrink = 0.05 * n / n_nodes
node_v = (num + shrink * v.mean()) / (den + shrink)
inner = slice(2, n_nodes - 2)
print(f"smoother on the pricing particles vs stored table (inner nodes): max |diff| = "
      f"{np.abs(node_v[inner] - stored[inner]).max():.4f} (variance {stored[inner].min():.3f}-{stored[inner].max():.3f})")
ordered = np.sort(y)
order = np.argsort(y)
sub = np.random.default_rng(0).choice(n, 4000, replace=False)
view = (ordered[int(0.005 * n)], ordered[int(0.995 * n)])            # central 99% of the particles
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].scatter(y[sub], v[sub], s=2, alpha=0.3, label="particles (4000 of them)")
axes[0].plot(nodes_y, stored, "r-", lw=2, label="E[v | ln X] (stored, 41 nodes)")
axes[0].axhline(v.mean(), color="k", ls="--", lw=1, label="mean variance")
axes[0].set(xlabel="ln X at t = 1", ylabel="v", xlim=view, ylim=(0, 0.8), title="Variance given the spot")
axes[1].plot(y[order], 100 * sigma_lv[order], label="local vol sigma_LV")
axes[1].plot(y[order], 100 * np.sqrt(np.maximum(cond_v[order], 1e-4)), label="sqrt(E[v | ln X])")
axes[1].set(xlabel="ln X at t = 1", ylabel="vol (%)", xlim=view, title="Leverage = local vol / sqrt(E[v | X])")
right = axes[1].twinx()
right.plot(y[order], lev_hand[order], "g", label="leverage L (right axis)")
right.axhline(1.0, color="g", ls=":", lw=1)
right.set_ylabel("L")
right.legend(fontsize=8, loc="upper right")
for ax in axes:
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
plt.show()

- On the downside the particles have a much **higher** variance on average (negative correlation: the spot fell
  *because* the variance rose). Around the money it is **lowest**: many paths with a variance near zero (Feller
  violated) barely moved. That is how Heston produces its skew and smile.
- The leverage is the ratio of what the market wants (local vol) to what Heston gives at that spot. Where the two
  agree, $L \approx 1$: Heston already explains the smile there. Elsewhere $L$ raises or lowers the vol, spot by spot.

### Step 4: the leverage surface
The leverage used by the particles at several times, as a function of the moneyness (median of $L$ in bins of $\ln X$).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for t in (0.1, 0.5, 1.0, 2.0, 3.0):
    c = int(round(t * STEPS)) - 1
    yy, ll = np.log(X[:, c - 1]) if c > 0 else np.zeros(N_PATHS), LEV[:, c]
    edges = np.quantile(yy, np.linspace(0.005, 0.995, 41))
    mid = 0.5 * (edges[1:] + edges[:-1])
    med = [np.median(ll[(yy >= a) & (yy < b)]) for a, b in zip(edges[:-1], edges[1:])]
    ax.plot(mid, med, label=f"t = {t}")
ax.axhline(1.0, color="k", lw=0.8)
ax.set(xlabel="ln X (forward log-moneyness)", ylabel="leverage L", title="Leverage by time and moneyness")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

- **Short times** ($t = 0.1$): a strong smile in the leverage, up to 2 on the downside and 1.5 on the upside. The market's
  short-dated smile (steep skew and high wings, the 1M fit errors of `heston.ipynb`) is far stronger than Heston's, and
  the leverage supplies the difference.
- **From 1Y**: the leverage is above 1 around the money and below 1 in the wings. At long maturities Heston has too
  much curvature (its vol of vol is pushed up by the short wings: its 2Y–3Y smiles are too high in the wings and too
  low at the money), and the leverage takes it out.

The leverage is the part of the smile that the stochastic variance does not explain.

### Step 5: the smile repriced (Gyöngy at work)
Calls on the particles back to implied vols at the market strikes, against the market, local vol (same Monte Carlo
setup) and the Heston closed form.

In [ ]:
S_lv = lv.evolve(grid, dw[..., 0]).numpy()                        # local vol on the same draws (first normal)
rows = []
for T in (0.5, 1.0, 2.0, 3.0):
    c = int(round(T * STEPS)) - 1                                     # column of time T
    ks =K_NODES / float(eq.forward(T))
    mkt_iv = np.array([float(vs.volatility(K, T)) for K in K_NODES])
    slv_iv = mc_smile(X[:, c], ks, T)
    lv_iv = mc_smile(S_lv[:, c] / float(eq.forward(T)), ks, T)
    hes_iv = np.array([heston.implied_vol(K, T) for K in K_NODES])
    for name, ivs in (("SLV", slv_iv), ("local vol", lv_iv), ("Heston", hes_iv)):
        rows.append({"T": T, "model": name, **{f"{K / S0:.0%}": 100 * (a - b) for K, a, b in zip(K_NODES, ivs, mkt_iv)}})
print("model - market implied vol (vol points), by strike / spot:")
print(pd.DataFrame(rows).set_index(["T", "model"]).round(2).to_string())

SLV reprices the surface within a few tenths of a vol point from 1Y, like local vol, where Heston alone misses by
up to a vol point (and by many points at 1M–3M, not shown). The leverage did its job. One exception: the short-dated
downside (6M, 60–70%), where SLV is up to about one vol point high; section 6 shows that it is not Monte Carlo noise.

## 4. The mixing factor: one dial for the dynamics
The vanillas fix the marginals, not the dynamics: for every $\eta$ the model reprices the same surface. What changes is
how the smile evolves. The **forward smile** from 1Y to 2Y (the smile of $S_2 / S_1$, as in `heston.ipynb`) for several
$\eta$, with local vol and Heston as the two ends, and the 1Y repricing error for each $\eta$.

In [ ]:
ks = np.linspace(0.7, 1.3, 13)
c1, c2 = STEPS - 1, 2 * STEPS - 1
dw2 = dw[:, : 2 * STEPS]
fwd = {}
rows = []
for eta in (1.0, 0.5, 0.25, 0.05):
    Xe = X[:, : 2 * STEPS] if eta == 1.0 else tq.HestonSLVModel(heston_model, mixing=eta).process(eq).simulate(grid[: 2 * STEPS], dw2, keep_all=False)[0].numpy()
    fwd[f"SLV eta = {eta}"] = mc_smile(Xe[:, c2] / Xe[:, c1], ks, 1.0)
    k1 = K_NODES / float(eq.forward(1.0))
    err = 100 * (mc_smile(Xe[:, c1], k1, 1.0) - np.array([float(vs.volatility(K, 1.0)) for K in K_NODES]))
    rows.append({"eta": eta, "1Y repricing: mean |error| (vol pts)": np.nanmean(np.abs(err))})
Xh = heston.simulate(grid[: 2 * STEPS], dw2)[0].numpy()
fwd["Heston"] = mc_smile(Xh[:, c2] / Xh[:, c1], ks, 1.0)
fwd["local vol"] = mc_smile((S_lv[:, c2] / S_lv[:, c1]) / (float(eq.forward(2.0)) / float(eq.forward(1.0))), ks, 1.0)
today = np.array([float(vs.volatility(k * float(eq.forward(1.0)), 1.0)) for k in ks])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, 100 * today, "k-o", ms=3, label="today's 1Y smile (market)")
for name, s in fwd.items():
    ax.plot(ks, 100 * s, "--" if name in ("Heston", "local vol") else "-", label=name)
ax.set(xlabel="k (strike / forward)", ylabel="implied vol (%)", title="Forward smile 1Y -> 2Y")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()
i100 = int(np.argmin(np.abs(ks - 1.0)))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(pd.DataFrame({name: {"ATM": 100 * s[i100], "70% - ATM": 100 * (s[0] - s[i100])} for name, s in fwd.items()}).T.round(1).to_string())

- Every $\eta$ reprices the 1Y smile (the errors stay at the Monte Carlo level).
- The forward smile moves continuously from the flat local vol one ($\eta \to 0$) towards the Heston one ($\eta = 1$; not
  exactly Heston's, because the leverage also acts after 1Y).
- The vanillas cannot tell which $\eta$ is right. In practice $\eta$ is set so that the model reprices products that
  depend on the dynamics, when quotes exist (forward-start options, barriers, cliquets), or by judgement; values well
  below 1 are common.

## 5. An exotic under the whole family
A 1Y down-and-in put (strike 100%, barrier 80%, monthly monitoring): all the models below reprice the same vanillas
(Heston approximately), so the differences are pure **dynamics**, i.e. model risk.

In [ ]:
expiry = date_of(1.0)
months = [date_of(m / 12) for m in range(1, 13)]
dip = lambda: tq.BarrierOption(eq.currency, EVALUATION_DATE, expiry, tq.OptionType.Put, S0, 0.8 * S0,
                               tq.BarrierType.DownIn, monitoring_dates=months, underlying=TICKER)
# model instances (Heston calibrated once above): the pricer binds each one to the equity
models = {"local vol": tq.LocalVolatilityModel(), **{f"SLV eta = {e}": tq.HestonSLVModel(heston_model, mixing=e)
                                                    for e in (0.25, 0.5, 1.0)},
          "Heston": heston_model}
rows = []
for name, model in models.items():
    product = dip()
    tq.VanillaMCPricer(model=model, n_paths=40_000, n_steps=252, seed=11).price(product, mkt, autodiff=False)
    rows.append({"model": name, "price (% spot)": 100 * float(product.price) / S0,
                 "s.e.": 100 * float(product.standard_error) / S0})
print(pd.DataFrame(rows).set_index("model").round(3).to_string())

All the models are on the same seed, so the differences are more precise than the standard errors suggest. The
effect of the dynamics is modest for this product (about 0.3% of the spot between local vol and SLV with $\eta = 1$): a
1Y put knocked in at 80% is dominated by the 1Y distribution, which all of them share; products that depend more on
the path and on future smiles (forward starts, cliquets, autocallables with many dates) show larger gaps (section 4).
Heston also differs because it does not reprice the vanillas exactly.

## 6. Monte Carlo

### How well the vanillas are repriced: noise and a systematic error
$\mathbb{E}[v \mid X]$ is estimated from a finite cloud of calibration particles, so the leverage is noisy and the repriced smile
carries Monte Carlo error. Step 5 also showed something else: at 6M the SLV smile is too high on the downside, where
local vol on the same draws is not. Is it noise? Below, the 6M error at the market strikes (200k pricing paths),
averaged over two calibration seeds, for an increasing number of calibration particles; then, with 200k particles,
for decreasing mixing factors (with local vol on the same draws as the $\eta \to 0$ reference).

In [ ]:
T6 = 0.5
g6 = np.arange(1, int(T6 * STEPS) + 1) / STEPS
k6 = K_NODES / float(eq.forward(T6))
mkt6 = np.array([float(vs.volatility(K, T6)) for K in K_NODES])
cols = [f"{K / S0:.0%}" for K in K_NODES]
rows = {}
d = tf.random.stateless_normal([200_000, len(g6), 2], seed=[0, 9], dtype=tf.float64)
for n_p in (20_000, 80_000, 320_000):
    errs = []
    for seed in (1, 2):
        m = tq.HestonSLVModel(heston_model, n_particles=n_p, seed=seed, horizon=T6)
        errs.append(100 * (mc_smile(m.process(eq).simulate(g6, d, keep_all=False)[0].numpy()[:, -1], k6, T6) - mkt6))
    rows[f"{n_p} particles"] = np.mean(errs, axis=0)
for eta in (1.0, 0.25, 0.05):
    m = tq.HestonSLVModel(heston_model, mixing=eta, n_particles=200_000, horizon=T6)
    x = m.process(eq).simulate(g6, d, keep_all=False)[0].numpy()[:, -1]
    rows[f"eta = {eta} (200k)"] = 100 * (mc_smile(x, k6, T6) - mkt6)
rows["local vol (200k)"] = 100 * (mc_smile(lv.evolve(g6, d[..., 0]).numpy()[:, -1], k6, T6) - mkt6)
print("6M: model - market implied vol (vol points), by strike / spot:")
print(pd.DataFrame(rows, index=cols).T.round(2).to_string())

- The **noise** falls with the number of particles, but the downside error does **not** vanish with them: it is
  systematic.
- As $\eta \to 0$ it disappears and SLV becomes local vol (same numbers, including local vol's own small discretisation
  error): the log step and the leverage are applied correctly; the error comes with the stochastic variance.
- Checked separately (on the earlier on-the-fly estimator, not shown): it did not change with 400 smoother nodes
  instead of 100, without the shrinkage, with 200 steps a year instead of 50, or without the cap on $L$ and the floor on
  $\mathbb{E}[v \mid X]$, and it faded with the maturity. The cause is not identified: an open point of the library.

Practical consequences: on the short-dated downside SLV does not reprice the vanillas as well as local vol (here
under one vol point), and a product that lives there (a short down-and-in put, a digital) inherits it. As in
production implementations, the leverage is built once on a dedicated simulation (`calibrate_leverage`) and the
products are priced on fresh paths.

### Greeks by AAD and the risk convention
The convention: **the Heston parameters are held** and the leverage moves with the market through the local vol (the
vol nodes), the forward (spot, repo, dividends, curve) **and the table** $\mathbb{E}[v \mid X]$, which is recalibrated when the market
moves. The table's channel goes through its calibration (Savine's approach: model greeks, then the adjoint of the
calibration):
- with a model instance the pricing tape gives the direct greeks and $\partial V / \partial \mathbb{E}$; `tq.par_greeks(model, ..., equity=eq)`
  re-runs the leverage calibration inside a tape and maps $\partial V / \partial \mathbb{E}$ to the market in one backward pass (for a
  portfolio, the trades' $\partial V / \partial \mathbb{E}$ summed, one pass);
- `VanillaMCPricer(model=HestonSLVModel)` calibrates the leverage inside the pricing tape: the same greeks in one pass.
- Without the table's channel the vega is wrong: on the snapshot a 1Y 80% put had 0.45 per vol point on its node instead
  of about 0.55, the rest on other strikes of the same expiry, and a 5Y one only 1.10 of total vega instead of 1.50
  (`tests/test_slv_vega.py` checks the adjoint against bumps of the two-stage model, at 1Y and 5Y).
- The adjoint needs a smooth estimator: its exact derivative follows the particle noise of each node, which grows
  along the calibration. 41 nodes are smooth enough (and end nodes off the particles: a node on a particle is a kink
  exactly at the calibration point); 101 nodes, or the Richardson-corrected estimate, are not (the adjoint blows up).
  Hence Savine's *smoothing for risk*: the price from the corrected table, the greeks from the 41-node one (the paths
  are `X_risk + stop_gradient(X_price - X_risk)`). Against bumps of the corrected model: within 3% on the node and on
  the parallel vega, up to 0.05 per vol point on other nodes (the greeks of a slightly smoother model).
- In exact SLV a vanilla does not depend on the Heston parameters at all (the leverage compensates them), so holding
  them loses nothing for vanillas; for an exotic, a recalibration of Heston would move the dynamics: that is a separate
  model-risk number.

Below, a 1Y ATM call: since SLV reprices the vanillas, its delta and vega per node should be close to the local vol
ones (and to the Black–Scholes sticky-strike delta with the same cash dividends).

In [ ]:
K = round(float(eq.forward(expiry)), 2)


def call():
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=tq.OptionType.Call,
                            strike=K, underlying=TICKER)


sources = [eq.spot, vs.volatility_matrix]
out = {}
for name, pricer in (("SLV", tq.VanillaMCPricer(model=tq.HestonSLVModel, n_paths=40_000, n_steps=50, seed=7)),
                     ("local vol", tq.VanillaMCPricer(model=tq.LocalVolatilityModel, n_paths=40_000, n_steps=50, seed=7)),
                     ("Black-Scholes", tq.BlackScholesPricer(dividend_model="discrete"))):
    product = call()
    pricer.price(product, mkt, autodiff=True)
    d, vg = pricer.tape.gradient(product.price, sources)
    out[name] = (float(product.price), float(getattr(product, "standard_error", 0.0)), float(d), vg.numpy() / 100)
for name, (pr, se, d, vg) in out.items():
    print(f"{name:14s} price {pr:7.3f} +/- {se:.3f}   delta {d:.4f}   vega (sum, per vol pt) {vg.sum():.3f}")
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4), sharey=True)
for ax, (name, (_, _, _, vg)) in zip(axes, out.items()):
    im = ax.imshow(vg, aspect="auto", cmap="RdBu_r", vmin=-0.4, vmax=0.4)
    ax.set(title=f"vega per node: {name}", xticks=range(len(K_NODES)), xticklabels=[f"{k:.0%}" for k in K_NODES / S0],
           yticks=range(len(T_NODES)), yticklabels=np.round(T_NODES, 2), xlabel="K / S0")
axes[0].set_ylabel("T")
fig.colorbar(im, ax=axes)
plt.show()

The SLV vega sits where the local vol vega sits (on the 1Y nodes around the strike, small contributions on the
neighbours), not spread over the surface as in Heston: with the Heston parameters held, the vol nodes act only through
the local vol and the table, i.e. through the marginals.

## 7. The limits of SLV
SLV fixes the two main limits of its components (local vol's dynamics, Heston's fit), at a price.

**1. The dynamics is a choice, not a calibration.** The vanillas do not determine $\eta$ (nor the Heston parameters'
role in the dynamics): two SLV models with different $\eta$ reprice the same surface and give different exotic prices
(section 5). Without quotes of forward-smile products, $\eta$ is judgement, and the exotic price carries that model risk.

**2. The particle estimator and the vanilla fit.** $\mathbb{E}[v \mid X]$ is estimated on a finite cloud: Monte Carlo noise on
the vanillas, choices (nodes, kernel, shrinkage, floors, the cap on $L$) that matter in the wings where few particles
are; and here a systematic error on the short-dated downside (section 6), not explained yet.

**3. Cost.** A leverage calibration (a few seconds) whenever the surface moves, after a Heston calibration now and then;
the market greeks need the adjoint of that calibration (one backward pass per underlying and portfolio; long
maturities cost most). Slower than local vol or Heston alone.

**4. Greeks by convention.** The Heston parameters are held: the greeks are those of a model whose dynamics is frozen
while the marginals follow the market. A sensible convention, but a convention: the sensitivity to the dynamics has to
be measured separately (bump $\eta$, recalibrate Heston).

**5. It inherits from both components.** From local vol: the dependence on the interpolation of the surface and the
noise of Dupire in the wings. From Heston: Feller violated (the QE scheme), and a vol of vol fitted to today's smile
that, scaled by $\eta$, sets the forward smile.

**6. Extreme leverage.** Where Heston and the market disagree most (short maturities, deep wings), $L$ is far from 1;
the leverage then does most of the work and the model behaves locally like local vol.

| limit | consequence | remedy |
|---|---|---|
| $\eta$ not identified by vanillas | exotic prices depend on a judgement | calibrate $\eta$ to forward-smile products, report the range |
| particle noise, short-dated error | vanilla errors | more calibration particles, a PDE for the forward density |
| cost | leverage calibration, adjoint for the greeks | calibrate once, reuse across products, one adjoint per portfolio |
| greeks by convention | dynamics risk not in the greeks | bump $\eta$ / the Heston parameters separately |
| inherited limits | interpolation, Feller | good surface construction, QE |

In short: SLV is the standard equity model for path-dependent exotics (barriers, autocallables, cliquets), because it
reprices the hedging vanillas and lets the desk choose the dynamics; the cost is a slower, noisier Monte Carlo and a
parameter ($\eta$) that the vanillas cannot pin down.

## Exercises
1. Repeat step 3 at $t = 0.1$: how far from 1 is the leverage in the wings, and why?
2. Switch the correction off (`tq.HestonSLVModel(heston_model, bias_correction=False)`) and replace the 41 nodes by
   21 and by 101: what happens to the 3Y repricing error with 20k calibration particles?
3. Price the down-and-in put of section 5 with continuous monitoring (no `monitoring_dates`) for $\eta = 0.25$ and $1$: does
   the model gap grow?